# T20 World Cup Match Insights & Analysis

## Comprehensive match-level analysis across T20 World Cups (2014-2024)

This notebook explores:
- Winning factors and patterns
- Toss impact on match outcomes
- Team performance comparisons
- Venue analysis
- Tournament trends over time
- Match predictions and key insights

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print("✅ Libraries loaded")

In [ ]:
# Load all datasets
matches = pd.read_csv('../data/processed/match_summaries.csv')
deliveries = pd.read_csv('../data/processed/all_deliveries.csv')
batting_stats = pd.read_csv('../data/processed/player_batting_stats.csv')
bowling_stats = pd.read_csv('../data/processed/player_bowling_stats.csv')

print(f"📊 Loaded data:")
print(f"  - Matches: {len(matches)}")
print(f"  - Deliveries: {len(deliveries):,}")
print(f"  - Batsmen: {len(batting_stats)}")
print(f"  - Bowlers: {len(bowling_stats)}")

## 1. Toss Analysis - Does Winning Toss Matter?

In [ ]:
# Analyze toss impact
matches_with_result = matches[matches['outcome_winner'].notna()].copy()
matches_with_result['toss_winner_won'] = matches_with_result['toss_winner'] == matches_with_result['outcome_winner']

toss_wins = matches_with_result['toss_winner_won'].sum()
toss_losses = len(matches_with_result) - toss_wins
toss_win_pct = (toss_wins / len(matches_with_result) * 100)

print("🎲 TOSS IMPACT ANALYSIS")
print("=" * 60)
print(f"Total matches with results: {len(matches_with_result)}")
print(f"Toss winner also won match: {toss_wins} ({toss_win_pct:.1f}%)")
print(f"Toss winner lost match: {toss_losses} ({100-toss_win_pct:.1f}%)")
print(f"\n💡 Insight: Winning the toss gives a {toss_win_pct:.1f}% chance of winning")

In [ ]:
# Toss decision analysis
print("\n🏏 TOSS DECISION ANALYSIS")
print("=" * 60)
toss_decisions = matches['toss_decision'].value_counts()
print("\nToss decisions:")
for decision, count in toss_decisions.items():
    pct = (count / len(matches) * 100)
    print(f"  {decision.capitalize():8s}: {count:3d} ({pct:5.1f}%)")

# Winning after choosing to bat/field
for decision in ['bat', 'field']:
    subset = matches_with_result[matches_with_result['toss_decision'] == decision]
    won = subset['toss_winner_won'].sum()
    total = len(subset)
    win_pct = (won / total * 100) if total > 0 else 0
    print(f"\nWin rate after choosing to {decision}: {won}/{total} ({win_pct:.1f}%)")

In [ ]:
# Visualization: Toss impact
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart: Toss winner outcome
labels = ['Won Match', 'Lost Match']
sizes = [toss_wins, toss_losses]
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 12})
ax1.set_title('Toss Winner Match Outcome', fontsize=14, fontweight='bold')

# Bar chart: Toss decision
toss_decisions.plot(kind='bar', ax=ax2, color=['steelblue', 'coral'], edgecolor='black')
ax2.set_title('Toss Decisions: Bat vs Field', fontsize=14, fontweight='bold')
ax2.set_xlabel('Decision', fontsize=12)
ax2.set_ylabel('Number of Matches', fontsize=12)
ax2.set_xticklabels(['Bat First', 'Field First'], rotation=0)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Team Performance Analysis

In [ ]:
# Extract all teams
all_teams = []
for teams_list in matches['teams']:
    if isinstance(teams_list, list):
        all_teams.extend(teams_list)

# Count wins per team
wins = matches['outcome_winner'].value_counts()
team_matches = Counter(all_teams)

# Create team performance dataframe
team_perf = pd.DataFrame({
    'team': list(team_matches.keys()),
    'matches': list(team_matches.values())
})
team_perf['wins'] = team_perf['team'].map(wins).fillna(0).astype(int)
team_perf['losses'] = team_perf['matches'] - team_perf['wins']
team_perf['win_pct'] = (team_perf['wins'] / team_perf['matches'] * 100).round(1)
team_perf = team_perf.sort_values('wins', ascending=False)

print("🏆 TEAM PERFORMANCE RANKINGS")
print("=" * 80)
print(team_perf.head(15).to_string(index=False))

In [ ]:
# Visualization: Team performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Top 12 teams by wins
top_teams = team_perf.head(12)
x = range(len(top_teams))
ax1.bar(x, top_teams['wins'], label='Wins', color='#2ecc71', alpha=0.8)
ax1.bar(x, top_teams['losses'], bottom=top_teams['wins'], label='Losses', color='#e74c3c', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(top_teams['team'], rotation=45, ha='right')
ax1.set_ylabel('Number of Matches', fontsize=12)
ax1.set_title('Top 12 Teams - Wins vs Losses', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Win percentage for teams with 10+ matches
qualified_teams = team_perf[team_perf['matches'] >= 10].head(12)
colors = ['darkgreen' if pct >= 60 else 'steelblue' if pct >= 50 else 'coral' 
         for pct in qualified_teams['win_pct']]
ax2.barh(range(len(qualified_teams)), qualified_teams['win_pct'], color=colors)
ax2.set_yticks(range(len(qualified_teams)))
ax2.set_yticklabels(qualified_teams['team'])
ax2.set_xlabel('Win Percentage', fontsize=12)
ax2.set_title('Win Percentage (min 10 matches)', fontsize=14, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3, axis='x')

# Add percentage labels
for i, pct in enumerate(qualified_teams['win_pct']):
    ax2.text(pct + 1, i, f"{pct:.1f}%", va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Venue Analysis

In [ ]:
# Analyze venues
venue_counts = matches['venue'].value_counts().head(15)

print("🏟️  MOST USED VENUES")
print("=" * 60)
for venue, count in venue_counts.items():
    print(f"  {venue[:50]:50s}: {count:2d} matches")

# Calculate average scores per venue (simplified)
print("\n📊 Note: Detailed venue scoring analysis would require innings data")

## 4. Tournament Trends Over Time

In [ ]:
# Extract year from match_date
matches['year'] = pd.to_datetime(matches['match_date'], errors='coerce').dt.year

# Group by tournament year
yearly_stats = matches.groupby('year').agg({
    'match_id': 'count',
    'venue': 'nunique'
}).rename(columns={'match_id': 'matches', 'venue': 'venues'})

print("📅 TOURNAMENT TIMELINE")
print("=" * 60)
print(yearly_stats.to_string())

# Identify tournament years
print("\n🏆 Tournament Years:")
tournament_years = yearly_stats[yearly_stats['matches'] > 20].index.tolist()
for year in sorted(tournament_years):
    count = yearly_stats.loc[year, 'matches']
    print(f"  {year}: {count} matches")

## 5. Match Outcomes Analysis

In [ ]:
# Analyze winning margins
outcome_by = []
for by_dict in matches['outcome_by']:
    if isinstance(by_dict, dict):
        if 'runs' in by_dict:
            outcome_by.append(('runs', by_dict['runs']))
        elif 'wickets' in by_dict:
            outcome_by.append(('wickets', by_dict['wickets']))

outcome_df = pd.DataFrame(outcome_by, columns=['type', 'margin'])

print("🎯 MATCH OUTCOMES")
print("=" * 60)
print(f"Wins by runs: {len(outcome_df[outcome_df['type'] == 'runs'])}")
print(f"Wins by wickets: {len(outcome_df[outcome_df['type'] == 'wickets'])}")

if len(outcome_df[outcome_df['type'] == 'runs']) > 0:
    print(f"\nAverage winning margin (runs): {outcome_df[outcome_df['type'] == 'runs']['margin'].mean():.1f}")
if len(outcome_df[outcome_df['type'] == 'wickets']) > 0:
    print(f"Average winning margin (wickets): {outcome_df[outcome_df['type'] == 'wickets']['margin'].mean():.1f}")

## 6. Key Match Insights Summary

In [ ]:
print("="*80)
print("🏏 KEY MATCH INSIGHTS - T20 WORLD CUPS (2014-2024)")
print("="*80)

print(f"\n📊 Tournament Overview:")
print(f"  Total matches: {len(matches)}")
print(f"  Unique teams: {len(team_perf)}")
print(f"  Unique venues: {matches['venue'].nunique()}")
print(f"  Tournament editions: {len(tournament_years)}")

print(f"\n🏆 Most Successful Team:")
top_team = team_perf.iloc[0]
print(f"  {top_team['team']}: {top_team['wins']} wins in {top_team['matches']} matches ({top_team['win_pct']:.1f}%)")

print(f"\n🎲 Toss Impact:")
print(f"  Toss winners won match: {toss_win_pct:.1f}%")
print(f"  Teams prefer to: {toss_decisions.index[0]} first ({(toss_decisions.iloc[0]/len(matches)*100):.1f}%)")

print(f"\n🏟️  Most Used Venue:")
print(f"  {venue_counts.index[0][:60]}: {venue_counts.iloc[0]} matches")

print("\n" + "="*80)
print("\n💡 Analysis Complete! Key patterns identified for further investigation.")

## Next Steps

- Build predictive models for match outcomes
- Analyze first innings vs chase success rates
- Study impact of specific players on match results
- Create win probability models based on match situations